In [ ]:
# OTHELLO bootstrap: make the package importable from notebooks/
import sys
from pathlib import Path
_repo_root = Path.cwd().parent.resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [1]:
from othello.pipelines.ask_verification import build_verification_graph
import pandas as pd
from collections import Counter
import json
import os

DOWNSAMPLE=True

/Users/cameronkruger/OTHELLO-1/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from typing import Dict, List
import pandas as pd

def run_verification_on_datasets(
    datasets: Dict[str, pd.DataFrame],
    question_col: str = "question",
    answer_col: str = "answer",
    build_graph_fn=None,
    show_progress: bool = True,
) -> Dict[str, List[str]]:
    """
    Runs the verification graph on each row of each dataframe.
    Returns: {dataset_name: [verdicts...]} in row order.
    """
    if build_graph_fn is None:
        from othello.pipelines.ask_verification import build_verification_graph
        build_graph_fn = build_verification_graph

    graph = build_graph_fn()
    results: Dict[str, List[str]] = {}

    for name, df in datasets.items():
        verdicts: List[str] = []
        iterator = df.itertuples(index=False)
        if show_progress:
            try:
                from tqdm import tqdm
                iterator = tqdm(list(iterator), desc=f"Verifying {name}")
            except Exception:
                pass

        for row in iterator:
            row_dict = row._asdict()
            state = {
                "question": row_dict.get(question_col, ""),
                "answer": row_dict.get(answer_col, ""),
            }
            try:
                out = graph.invoke(state)
                verdicts.append(out.get("verdict", "Not Found"))
            except Exception:
                verdicts.append("ERROR")

        results[name] = verdicts

    return results


In [3]:
names = ['mintaka', 'qald', 'hotpot']

variants = ['small','base','large']
norm_files = [f'flan-t5-{variant}' for variant in variants]
vanilla_files = [f'vanilla-flan-t5-{variant}' for variant in variants]

files = norm_files + vanilla_files

file_name = files[0]


dataframes = {name: pd.read_csv(f'../data/llm_answers/{file_name}/LLM_Answers_{name}.csv') for name in names}

if file_name.startswith('vanilla'):
    dataframes = {name: pd.read_csv(f'../data/llm_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv') for name in names}




In [4]:
dataframes['qald']

,Unnamed: 0,AAVE Question,SAE Question,SPARQL,Results,Attempts,names,Answer
0,0,What time zone Salt Lake City in?,What time zone is Salt Lake City in?,SELECT ?timeZone ?timeZoneLabel WHERE {\n wd:...,"{'head': {'vars': ['timeZone', 'timeZoneLabel'...",1,UTC−07:00,UTC07:00
1,1,Who killed Caesar?,Who killed Caesar?,SELECT ?killer ?killerLabel WHERE {\n wd:Q104...,"{'head': {'vars': ['killer', 'killerLabel']}, ...",1,Marcus Junius Brutus,Caesar
2,2,Wat's da highest mountain in Germany?,What is the highest mountain in Germany?,SELECT ?item ?itemLabel ?elevation WHERE {\n ...,"{'head': {'vars': ['item', 'itemLabel', 'eleva...",1,Q136722681,I don't know
3,3,Which American presidents was in office durin'...,Which American presidents were in office durin...,SELECT DISTINCT ?person ?personLabel WHERE {\n...,"{'head': {'vars': ['person', 'personLabel']}, ...",1,John F. Kennedy,John F. Kennedy
4,4,Butch Otter the governor of which U.S. state?,Which U.S. state was Butch Otter the governor of?,SELECT DISTINCT ?state ?stateLabel WHERE {\n ...,"{'head': {'vars': ['state', 'stateLabel']}, 'r...",1,Idaho,Idaho
...,...,...,...,...,...,...,...,...
95,95,Where Piccadilly start at?,Where does Piccadilly start?,NaN,{},0,NaN,I don't know
96,96,What da name of da university where Obama's wi...,What is the name of the university where Obama...,NaN,{},0,NaN,I don't know
97,97,When Paraguay proclaim it independence?,When did Paraguay proclaim its independence?,NaN,{},0,NaN,I don't know
98,98,How short da shortest active NBA player?,How tall is the shortest active NBA player?,NaN,{},0,NaN,I am tall


In [13]:
from othello.pipelines.ask_verification import build_verification_graph
row = dataframes["qald"].iloc[0]
state = {"question": row["SAE Question"], "answer": row["Answer"]}
graph = build_verification_graph(state)



out = graph.invoke(state)



Rephrased: 'UTC07:00' -> 'Salt Lake City is in the UTC07:00 time zone.'
Split into 1 claims: ['Salt Lake City is in the UTC07:00 time zone']
Claim 0: 'Salt Lake City is in the UTC07:00 time zone' -> Entities: ['Salt Lake City', 'UTC07:00']
Claim 0: Relations: ['time zone']
Entity 'Salt Lake City' -> Q23337 (Salt Lake City)
Property 'time zone' -> P421 (located in time zone)
Skipped 1 queries: [(0, 'Insufficient entities (1)')]
Claim 0 verdict: NOT_FOUND - 'Salt Lake City is in the UTC07:00 time zone'
Overall verdict: Not Found


In [11]:
all_results = {}

for file_name in files:
    is_vanilla = file_name.startswith("vanilla")
    
    if is_vanilla:
        dataframes = {
            name: pd.read_csv(f"../data/llm_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv")
            for name in names
        }
    else:
        dataframes = {
            name: pd.read_csv(f"../data/llm_answers/{file_name}/LLM_Answers_{name}.csv")
            for name in names
        }

    file_results = {}

    for name, data in dataframes.items():
        print(f"Starting verification for {name} ({'Vanilla' if is_vanilla else 'Our Pipeline'})")
        if DOWNSAMPLE:
            print(f"Truncating {name} to 50 items")
            data = data[:50]

        row_results = []
        factual_answers = 0
        hallucinations = 0
        valid_count = 0
        
        # Store examples for the paper
        factual_examples = []
        hallucination_examples = []

        for idx, row in data.iterrows():
            state = {"question": row["SAE Question"], "answer": row["Answer"]}
            print(f"state:\t{state}")
            out = graph.invoke(state)
            results = out.get("results") or []

            valid = [r for r in results if r is not None]
            if not valid:
                row_results.append(None)
                continue

            is_factual = any(valid)
            row_results.append(is_factual)

            if is_factual:
                factual_answers += 1
                if len(factual_examples) < 3:
                    factual_examples.append({
                        "question": row["SAE Question"],
                        "answer": row["Answer"]
                    })
            else:
                hallucinations += 1
                if len(hallucination_examples) < 3:
                    hallucination_examples.append({
                        "question": row["SAE Question"],
                        "answer": row["Answer"]
                    })
            
            valid_count += 1

        # Calculate rates
        if valid_count > 0:
            factual_rate = (factual_answers / valid_count) * 100
            hallucination_rate = (hallucinations / valid_count) * 100
        else:
            factual_rate = None
            hallucination_rate = None

        print(
            f"{name} ({'Vanilla' if is_vanilla else 'Our Pipeline'}) — "
            f"Factual: {factual_answers}/{valid_count} ({factual_rate:.2f}%), "
            f"Hallucinations: {hallucinations}/{valid_count} ({hallucination_rate:.2f}%)"
        )

        file_results[name] = {
            "method": "vanilla" if is_vanilla else "pipeline",
            "raw_data": {
                "factual_count": factual_answers,
                "hallucination_count": hallucinations,
                "total_verified": valid_count
            },
            "performance_metrics": {
                "factual_accuracy_pct": factual_rate,
                "hallucination_rate_pct": hallucination_rate
            },
            "sample_outputs": {
                "factual_examples": factual_examples,
                "hallucination_examples": hallucination_examples
            },
            "detailed_results": row_results
        }

    out_dir = f"../results/{file_name}"
    os.makedirs(out_dir, exist_ok=True)

    json_path = f"{out_dir}/{file_name}_results.json"
    with open(json_path, "w") as f:
        json.dump(file_results, f, indent=2)

    all_results[file_name] = file_results


print("COMPARATIVE ANALYSIS: Pipeline vs Vanilla")
print("="*70)

comparative_analysis = {}
vanilla_keys = [k for k in all_results.keys() if k.startswith("vanilla")]
pipeline_keys = [k for k in all_results.keys() if not k.startswith("vanilla")]

if vanilla_keys and pipeline_keys:
    vanilla_key = vanilla_keys[0]
    pipeline_key = pipeline_keys[0]
    
    for name in names:
        vanilla_data = all_results[vanilla_key].get(name)
        pipeline_data = all_results[pipeline_key].get(name)
        
        if vanilla_data and pipeline_data:
            v_factual = vanilla_data["performance_metrics"]["factual_accuracy_pct"]
            p_factual = pipeline_data["performance_metrics"]["factual_accuracy_pct"]
            
            v_halluc = vanilla_data["performance_metrics"]["hallucination_rate_pct"]
            p_halluc = pipeline_data["performance_metrics"]["hallucination_rate_pct"]
            
            if v_factual is not None and p_factual is not None:
                # Key metrics for your paper
                factual_improvement = p_factual - v_factual
                halluc_reduction = v_halluc - p_halluc
                relative_halluc_reduction = (halluc_reduction / v_halluc * 100) if v_halluc > 0 else 0
                
                comparative_analysis[name] = {
                    "factual_accuracy": {
                        "vanilla_pct": v_factual,
                        "pipeline_pct": p_factual,
                        "absolute_improvement_pct": factual_improvement,
                        "relative_improvement_pct": (factual_improvement / v_factual * 100) if v_factual > 0 else 0
                    },
                    "hallucination_rate": {
                        "vanilla_pct": v_halluc,
                        "pipeline_pct": p_halluc,
                        "absolute_reduction_pct": halluc_reduction,
                        "relative_reduction_pct": relative_halluc_reduction
                    },
                    "raw_counts": {
                        "vanilla": vanilla_data["raw_data"],
                        "pipeline": pipeline_data["raw_data"]
                    }
                }
                
                print(f"\n{name}:")
                print(f"  Vanilla Hallucination Rate: {v_halluc:.2f}%")
                print(f"  Pipeline Hallucination Rate: {p_halluc:.2f}%")
                print(f"  → Hallucination Reduction: {halluc_reduction:.2f}% (absolute)")
                print(f"  → Relative Reduction: {relative_halluc_reduction:.2f}%")
                print(f"  Factual Accuracy: {v_factual:.2f}% → {p_factual:.2f}% (+{factual_improvement:.2f}%)")

all_results["comparative_analysis"] = comparative_analysis

all_results_path = "../results/all_results.json"
with open(all_results_path, "w") as f:
    json.dump(all_results, f, indent=2)



Starting verification for mintaka (Our Pipeline)
Truncating mintaka to 50 items
state:	{'question': 'What is the seventh tallest mountain in North America?', 'answer': 'El Diente Peak'}
Rephrased: 'El Diente Peak' -> 'El Diente Peak is the seventh tallest mountain in North America.'
Split into 3 claims: ['El Diente Peak is a mountain', 'El Diente Peak is located in North America', 'El Diente Peak is the seventh tallest mountain in North America']
Claim 0: 'El Diente Peak is a mountain' -> Entities: ['El Diente Peak']
Claim 1: 'El Diente Peak is located in North America' -> Entities: ['El Diente Peak', 'North America']
Claim 2: 'El Diente Peak is the seventh tallest mountain in North America' -> Entities: ['El Diente Peak', 'North America']
Claim 0: Relations: ['instance of']
Claim 1: Relations: ['located in']
Claim 2: Relations: ['ranked as']
Query 1: Q5351150 --P276-- Q49
Skipped 2 queries: [(0, 'Insufficient entities (1)'), (2, 'No properties found')]
Query 1 result: False
Claim 0 ve

In [52]:
for k1 in all_results.keys():
    for k2 in all_results[k1].keys():
        # for k3 in all_results[k1][k2].keys():
        print(all_results[k1][k2])
    

[{'Results': ['False'], 'Majority': 'None'}, {'Results': ['None', 'False'], 'Majority': 'None'}, {'Results': ['None', 'False', 'None'], 'Majority': 'None'}, {'Results': ['None'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}]
[{'Results': ['True'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}, {'Results': ['None'], 'Majority': 'None'}, {'Results': [], 'Majority': 'None'}, {'Results': ['False'], 'Majority': 'None'}]
[{'Results': ['None', 'False'], 'Majority': 'None'}, {'Results': ['False', 'None'], 'Majority': 'None'}, {'Results': ['False', 'None'], 'Majority': 'None'}, {'Results': ['None'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}]
[{'Results': ['None'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}, {'Results': ['False'], 'Majority': 'None'}, {'Results': ['None', 'False'], 'Majority': 'None'}]
[{'Results': ['True'], 'Majority': 'F

In [54]:
# import json

with open(all_results_path, "w") as f:
    json.dump(all_results, f, indent=2)
